In [1]:
# Supress Warnings 
import warnings
warnings.filterwarnings('ignore')

# Import common GIS tools
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray as rio
import rasterio
from matplotlib.cm import RdYlGn,jet,RdBu

# Import Planetary Computer tools
import stackstac
import pystac_client
import planetary_computer 
from odc.stac import stac_load

In [2]:
# Define the bounding box for the entire data region using (Latitude, Longitude)

# New York City Region
# lower_left = (40.75, -74.01)
# upper_right = (40.88, -73.86)

# Montgomery, Maryland Region
lower_left = (38.94, -77.34)
upper_right = (39.25, -76.90)

In [3]:
# Calculate the bounds for doing an archive data search
# bounds = (min_lon, min_lat, max_lon, max_lat)
bounds = (lower_left[1], lower_left[0], upper_right[1], upper_right[0])

In [4]:
# Define the time window

# New York City Region
# time_window = "2021-07-01/2021-07-31"

# Montgomery, Maryland Region
time_window = "2022-07-15/2022-08-15"

In [5]:
stac = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

search = stac.search(
    bbox=bounds, 
    datetime=time_window,
    collections=["sentinel-2-l2a"],
    query={"eo:cloud_cover": {"lt": 30}},
)

In [6]:
items = list(search.get_items())
print('This is the number of scenes that touch our region:',len(items))

This is the number of scenes that touch our region: 15


In [7]:
signed_items = [planetary_computer.sign(item).to_dict() for item in items]

In [8]:
# Define the pixel resolution for the final product
# Define the scale according to our selected crs, so we will use degrees
resolution = 10  # meters per pixel 
scale = resolution / 111320.0 # degrees per pixel for crs=4326 

In [9]:
data = stac_load(
    items,
    bands=["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"],
    crs="EPSG:4326", # Latitude-Longitude
    resolution=scale, # Degrees
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bounds
)

In [10]:
# View the dimensions of our XARRAY and the loaded variables
# This insures we have the right coordinates and spectral bands in our xarray
display(data)

<xarray.Dataset> Size: 3GB
Dimensions:      (latitude: 3451, longitude: 4899, time: 7)
Coordinates:
  * latitude     (latitude) float64 28kB 39.25 39.25 39.25 ... 38.94 38.94 38.94
  * longitude    (longitude) float64 39kB -77.34 -77.34 -77.34 ... -76.9 -76.9
    spatial_ref  int32 4B 4326
  * time         (time) datetime64[ns] 56B 2022-07-19T15:49:51.024000 ... 202...
Data variables:
    B01          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B02          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B03          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B04          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B05          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B06          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B07          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B08          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B8A          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B09          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B11          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    B12          (time, latitude, longitude) uint16 237MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>

In [11]:
median = data.median(dim="time").compute()

In [12]:
filename = "Maryland_S2.tiff"

In [13]:
# Calculate the dimensions of the file
height = median.dims["latitude"]
width = median.dims["longitude"]

In [14]:
# Define the Coordinate Reference System (CRS) to be common Lat-Lon coordinates
# Define the tranformation using our bounding box so the Lat-Lon information is written to the GeoTIFF
gt = rasterio.transform.from_bounds(lower_left[1],lower_left[0],upper_right[1],upper_right[0],width,height)
median.rio.write_crs("epsg:4326", inplace=True)
median.rio.write_transform(transform=gt, inplace=True);

In [15]:
# Create the GeoTIFF output file using the defined parameters 
with rasterio.open(filename,'w',driver='GTiff',width=width,height=height,
                   crs='epsg:4326',transform=gt,count=12,compress='lzw',dtype='float64') as dst:
    dst.write(median.B01,1)
    dst.write(median.B02,2)
    dst.write(median.B03,3) 
    dst.write(median.B04,4)
    dst.write(median.B05,5)
    dst.write(median.B06,6)
    dst.write(median.B07,7) 
    dst.write(median.B08,8)
    dst.write(median.B8A,9) 
    dst.write(median.B09,10)
    dst.write(median.B11,11) 
    dst.write(median.B12,12)
    dst.close()